# Empirical Characterization of Oil–Sovereign Transmission

This notebook consolidates the empirical groundwork for the thesis. Three blocks:

1. **OVX-conditioned regressions** — Brent returns affect exporter CDS differently, and the effect amplifies under high OVX
2. **Futures basis regressions** — The oil futures term structure has differential impact on exporter vs. control CDS
3. **Jump characterization** — GARCH vs. GARCH-Jump LR tests on oil benchmarks, CDS, and MSCI series

---

## 0. Imports & Data Loading

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import datetime
import warnings
warnings.filterwarnings('ignore')

from linearmodels.panel import PanelOLS
from scipy.optimize import minimize
from scipy.stats import chi2, ttest_ind, mannwhitneyu, fisher_exact
from numba import njit

In [2]:
# ── Country groups ──────────────────────────────────────────
oil_exporters = [
    'Saudi Arabia', 'Abu Dhabi', 'Qatar', 'Colombia',
    'Mexico', 'Brazil', 'Egypt', 'Malaysia'
]
control_countries = [
    'Indonesia', 'Philippines', 'Turkey', 'Chile', 'China',
    'South Africa', 'South Korea', 'Thailand'
]
all_countries = oil_exporters + control_countries

In [3]:
# ── Load datasets ──────────────────────────────────────────
CDS_data  = pd.read_csv('../data/processed/CDS/Weekly_CDS.csv', parse_dates=['date'], index_col='date')
Oil_data  = pd.read_csv('../data/processed/Oil/oil_prices_datastream.csv', parse_dates=['date'], index_col='date')
Oil_fut   = pd.read_csv('../data/processed/Oil/oil_futures.csv', parse_dates=['date'], index_col='date')
Macro     = pd.read_csv('../data/processed/Macroeconomic_variables/macro_risk_variables.csv', parse_dates=['Date'], index_col='Date')
VIX       = pd.read_csv('../data/processed/Macroeconomic_variables/VIXCLS.csv', parse_dates=['Date'], index_col='Date')
OVX       = pd.read_csv('../data/processed/Macroeconomic_variables/OVXCLS.csv', parse_dates=['date'], index_col='date')
FX_data   = pd.read_csv('../data/processed/Macroeconomic_variables/Daily_FX_Rates.csv', parse_dates=['Date'], index_col='Date')
MSCI_data = pd.read_csv('../data/processed/MSCI_indices/mscicountryindex.csv', parse_dates=['date'], index_col='date')

print(f'CDS: {CDS_data.shape}, Oil: {Oil_data.shape}, Futures: {Oil_fut.shape}')

CDS: (1305, 72), Oil: (6524, 4), Futures: (6523, 5)


## 0.1 Build Panel

In [4]:
# ── Merge global variables ─────────────────────────────────
merged = (
    CDS_data
    .join(Oil_data, how='inner')
    .join(Oil_fut, how='inner')
    .join(Macro, how='inner')
    .join(VIX, how='inner')
    .join(OVX, how='inner')
)
merged = merged.loc[(merged.index >= '2014-01-01') & (merged.index <= '2024-12-31')].copy()

# ── Global transforms ──────────────────────────────────────
merged['Oil_ret']    = np.log(merged['Brent'] / merged['Brent'].shift(1))
merged['OVX_chg']    = np.log(merged['OVXCLS'] / merged['OVXCLS'].shift(1))
merged['VIX_chg']    = np.log(merged['VIXCLS'] / merged['VIXCLS'].shift(1))
merged['DXY_chg']    = np.log(merged['DXY'] / merged['DXY'].shift(1))
merged['UST10Y_chg'] = merged['UST10Y'].diff()

# ── Futures curve ratios ───────────────────────────────────
for tenor, col in [('1m','Brent_1m'),('3m','Brent_3m'),('6m','Brent_6m'),
                    ('12m','Brent_12m'),('24m','Brent_24m')]:
    if col in merged.columns:
        merged[f'log_curve_{tenor}']   = np.log(merged[col] / merged['Brent'])
        merged[f'd_log_curve_{tenor}'] = merged[f'log_curve_{tenor}'].diff()

# ── Reshape to long panel ──────────────────────────────────
global_cols = [c for c in merged.columns if c not in all_countries]
merged = merged.reset_index()

fx_aligned   = FX_data.reindex(pd.to_datetime(merged['date'])).reset_index()
msci_aligned = MSCI_data.reindex(pd.to_datetime(merged['date'])).reset_index()

panel_rows = []
for country in all_countries:
    if country not in merged.columns:
        print(f'Warning: {country} not in CDS data')
        continue
    temp = merged[['date'] + global_cols + [country]].copy() if country in merged.columns else None
    # Simpler: just grab global + CDS
    temp = merged[['date'] + [c for c in global_cols if c in merged.columns]].copy()
    temp['CDS'] = merged[country].values
    temp['Country'] = country
    temp['OilExporter'] = int(country in oil_exporters)
    if country in fx_aligned.columns:
        temp['FX'] = fx_aligned[country].values
    else:
        temp['FX'] = np.nan
    if country in msci_aligned.columns:
        temp['MSCI'] = msci_aligned[country].values
    else:
        temp['MSCI'] = np.nan
    panel_rows.append(temp)

panel = pd.concat(panel_rows, ignore_index=True).sort_values(['Country','date']).reset_index(drop=True)

# ── Country-specific log-returns ───────────────────────────
panel['CDS_ret']  = panel.groupby('Country')['CDS'].transform(lambda s: np.log(s / s.shift(1)))
panel['FX_ret']   = panel.groupby('Country')['FX'].transform(lambda s: np.log(s / s.shift(1)))
panel['MSCI_ret'] = panel.groupby('Country')['MSCI'].transform(lambda s: np.log(s / s.shift(1)))

# ── Interaction terms ──────────────────────────────────────
panel['Brent_x_Exporter'] = panel['Oil_ret'] * panel['OilExporter']

# ── Drop incomplete rows ──────────────────────────────────
panel = panel.dropna(subset=['CDS_ret','Oil_ret','VIX_chg','DXY_chg','UST10Y_chg']).reset_index(drop=True)

print(f'Panel: {panel.shape[0]} rows, {panel["Country"].nunique()} countries, '
      f'~{len(panel)//panel["Country"].nunique()} weeks each')

Panel: 8592 rows, 16 countries, ~537 weeks each


---

# 1. OVX-Conditioned Regressions

## 1.1 Panel regressions: Differential oil sensitivity

Four nested specifications testing whether $\beta_2$ on `Brent × Exporter` is negative and significant.

In [5]:
reg_panel = panel.copy()
reg_panel = reg_panel.set_index(['Country', 'date'])
y = reg_panel['CDS_ret']

In [6]:
# Model 1: Baseline
X_1 = reg_panel[['Oil_ret', 'Brent_x_Exporter']]
results_1 = PanelOLS(y, X_1, entity_effects=False, time_effects=False
            ).fit(cov_type='clustered', cluster_entity=True)

# Model 2: + controls
X_2 = reg_panel[['Oil_ret', 'Brent_x_Exporter', 'VIX_chg',
                 'DXY_chg', 'MSCI_ret', 'FX_ret', 'UST10Y_chg']]
results_2 = PanelOLS(y, X_2, entity_effects=False, time_effects=False
            ).fit(cov_type='clustered', cluster_entity=True)

# Model 3: + country FE
X_3 = reg_panel[['Oil_ret', 'Brent_x_Exporter', 'VIX_chg',
                 'DXY_chg', 'MSCI_ret', 'FX_ret', 'UST10Y_chg']]
results_3 = PanelOLS(y, X_3, entity_effects=True, time_effects=False
            ).fit(cov_type='clustered', cluster_entity=True)

# Model 4: + country FE + time FE
X_4 = reg_panel[['Brent_x_Exporter']]
results_4 = PanelOLS(y, X_4, entity_effects=True, time_effects=True
            ).fit(cov_type='clustered', cluster_entity=True)

In [7]:
# ── Summary table ─────────────────────────────────────────
print(f"\n{'Model':<45} {'β':>10} {'SE':>10} {'p-value':>10} {'R²':>10}")
print('-' * 85)

for name, res in [('Model 1: Baseline', results_1),
                   ('Model 2: + Global Controls', results_2),
                   ('Model 3: + Country FE', results_3),
                   ('Model 4: + Country FE + Time FE', results_4)]:
    beta = res.params['Brent_x_Exporter']
    se   = res.std_errors['Brent_x_Exporter']
    pval = res.pvalues['Brent_x_Exporter']
    r2   = res.rsquared_within if hasattr(res, 'rsquared_within') else res.rsquared
    sig  = '***' if pval<0.01 else '**' if pval<0.05 else '*' if pval<0.1 else ''
    print(f'{name:<45} {beta:>10.4f} {se:>10.4f} {pval:>10.4f} {r2:>10.4f} {sig}')

print('\n* p<0.1, ** p<0.05, *** p<0.01 — SEs clustered by country')


Model                                                  β         SE    p-value         R²
-------------------------------------------------------------------------------------
Model 1: Baseline                                -0.0713     0.0630     0.2575     0.0740 
Model 2: + Global Controls                       -0.0439     0.0458     0.3381     0.3542 
Model 3: + Country FE                            -0.0439     0.0458     0.3383     0.3542 
Model 4: + Country FE + Time FE                  -0.0713     0.0651     0.2734     0.0137 

* p<0.1, ** p<0.05, *** p<0.01 — SEs clustered by country


## 1.2 OVX threshold regressions

Restrict sample to weeks when OVX > quantile, re-estimate. If the exporter differential is tail-driven, $|\hat{\beta}_2|$ should increase.

In [8]:
results_list = []

ovx_quantiles = {
    'Full Sample': None,
    'OVX > Q70':   panel['OVXCLS'].quantile(0.70),
    'OVX > Q80':   panel['OVXCLS'].quantile(0.80),
    'OVX > Q90':   panel['OVXCLS'].quantile(0.90),
    'OVX > Q95':   panel['OVXCLS'].quantile(0.95),
    'OVX > Q99':   panel['OVXCLS'].quantile(0.99),
}

for label, cutoff in ovx_quantiles.items():
    subset = panel.copy() if cutoff is None else panel[panel['OVXCLS'] > cutoff].copy()
    subset = subset.set_index(['Country', 'date'])

    y_sub = subset['CDS_ret']
    X = subset[['Oil_ret', 'Brent_x_Exporter', 'VIX_chg',
                'DXY_chg', 'MSCI_ret', 'FX_ret', 'UST10Y_chg']]
    res = PanelOLS(y_sub, X, entity_effects=True, time_effects=False,
                   drop_absorbed=True, check_rank=False
          ).fit(cov_type='clustered', cluster_entity=True)

    results_list.append({
        'Threshold': label, 'N': len(y_sub),
        'β': res.params['Brent_x_Exporter'],
        'SE': res.std_errors['Brent_x_Exporter'],
        'p': res.pvalues['Brent_x_Exporter'],
    })

print(f"{'Threshold':<15} {'N':>6}  {'β':>8} {'p':>7}")
print('-' * 40)
for r in results_list:
    sig = '***' if r['p']<0.01 else '**' if r['p']<0.05 else '*' if r['p']<0.1 else ''
    print(f"{r['Threshold']:<15} {r['N']:>6}  {r['β']:>8.4f} {r['p']:>6.3f}{sig}")

Threshold            N         β       p
----------------------------------------
Full Sample       8592   -0.0439  0.338
OVX > Q70         2560   -0.0684  0.215
OVX > Q80         1712   -0.1138  0.045**
OVX > Q90          848   -0.1648  0.005***
OVX > Q95          416   -0.2366  0.000***
OVX > Q99           80   -0.3233  0.004***


---

# 2. Futures Basis Regressions

Test whether the oil futures term structure (log curve ratio at various tenors) affects CDS changes differently for exporters vs. controls.

## 2.1 Interaction specification (changes in curve ratio)

In [9]:
controls_list = ['VIX_chg', 'FX_ret', 'MSCI_ret', 'UST10Y_chg', 'DXY_chg']

for tenor, tlabel in [('d_log_curve_1m','1m'),('d_log_curve_3m','3m'),
                       ('d_log_curve_6m','6m'),('d_log_curve_12m','12m'),
                       ('d_log_curve_24m','24m')]:
    if tenor not in panel.columns:
        continue
    panel[f'Curve_{tlabel}_x_Exp'] = panel[tenor] * panel['OilExporter']

    ivs = [tenor, f'Curve_{tlabel}_x_Exp'] + controls_list
    sub = panel.dropna(subset=['CDS_ret'] + ivs).copy()
    sub = sub.set_index(['Country', 'date'])

    mod = PanelOLS(sub['CDS_ret'], sub[ivs], entity_effects=True
          ).fit(cov_type='clustered', cluster_entity=True)

    b_base = mod.params[tenor]
    p_base = mod.pvalues[tenor]
    b_int  = mod.params[f'Curve_{tlabel}_x_Exp']
    p_int  = mod.pvalues[f'Curve_{tlabel}_x_Exp']

    sb = '***' if p_base<0.01 else '**' if p_base<0.05 else '*' if p_base<0.1 else ''
    si = '***' if p_int<0.01  else '**' if p_int<0.05  else '*' if p_int<0.1  else ''

    print(f'Tenor {tlabel}:  Base β={b_base:.3f}{sb}  Interaction β={b_int:.3f}{si}  '
          f'Total(exp)={b_base+b_int:.3f}  R²={mod.rsquared:.4f}  N={int(mod.nobs)}')

Tenor 1m:  Base β=0.174***  Interaction β=-0.139*  Total(exp)=0.035  R²=0.3498  N=8592
Tenor 3m:  Base β=-0.037  Interaction β=0.146***  Total(exp)=0.109  R²=0.3498  N=8592
Tenor 6m:  Base β=-0.069*  Interaction β=0.156***  Total(exp)=0.087  R²=0.3501  N=8592
Tenor 12m:  Base β=-0.022  Interaction β=0.133***  Total(exp)=0.111  R²=0.3504  N=8592
Tenor 24m:  Base β=0.027  Interaction β=0.100**  Total(exp)=0.128  R²=0.3512  N=8592


## 2.2 Split-sample: exporters vs. controls separately

In [10]:
for tenor, tlabel in [('d_log_curve_1m','1m'),('d_log_curve_3m','3m'),
                       ('d_log_curve_6m','6m'),('d_log_curve_12m','12m'),
                       ('d_log_curve_24m','24m')]:
    if tenor not in panel.columns:
        continue
    ivs = [tenor] + controls_list

    print(f'\nTenor: {tlabel}')
    for group, glabel in [(1, 'Exporters'), (0, 'Controls')]:
        sub = panel[panel['OilExporter'] == group].dropna(subset=['CDS_ret'] + ivs).copy()
        sub = sub.set_index(['Country', 'date'])
        mod = PanelOLS(sub['CDS_ret'], sub[ivs], entity_effects=True
              ).fit(cov_type='clustered', cluster_entity=True)
        b = mod.params[tenor]
        p = mod.pvalues[tenor]
        sig = '***' if p<0.01 else '**' if p<0.05 else '*' if p<0.1 else ''
        print(f'  {glabel:<12} β={b:>7.3f}{sig:<3} p={p:.4f}  R²={mod.rsquared:.4f}  N={int(mod.nobs)}')


Tenor: 1m
  Exporters    β=  0.053    p=0.4361  R²=0.3166  N=4296
  Controls     β=  0.129*** p=0.0027  R²=0.4010  N=4296

Tenor: 3m
  Exporters    β=  0.128**  p=0.0226  R²=0.3174  N=4296
  Controls     β= -0.055    p=0.1625  R²=0.4008  N=4296

Tenor: 6m
  Exporters    β=  0.102**  p=0.0236  R²=0.3175  N=4296
  Controls     β= -0.081**  p=0.0103  R²=0.4013  N=4296

Tenor: 12m
  Exporters    β=  0.126*** p=0.0009  R²=0.3187  N=4296
  Controls     β= -0.032    p=0.2194  R²=0.4008  N=4296

Tenor: 24m
  Exporters    β=  0.142*** p=0.0000  R²=0.3204  N=4296
  Controls     β=  0.017    p=0.4783  R²=0.4007  N=4296


## 2.3 Level of curve ratio (split-sample)

In [11]:
for tenor, tlabel in [('log_curve_1m','1m'),('log_curve_3m','3m'),
                       ('log_curve_6m','6m'),('log_curve_12m','12m'),
                       ('log_curve_24m','24m')]:
    if tenor not in panel.columns:
        continue
    ivs = [tenor] + controls_list

    print(f'\nTenor: {tlabel}')
    for group, glabel in [(1, 'Exporters'), (0, 'Controls')]:
        sub = panel[panel['OilExporter'] == group].dropna(subset=['CDS_ret'] + ivs).copy()
        sub = sub.set_index(['Country', 'date'])
        mod = PanelOLS(sub['CDS_ret'], sub[ivs], entity_effects=True
              ).fit(cov_type='clustered', cluster_entity=True)
        b = mod.params[tenor]
        p = mod.pvalues[tenor]
        sig = '***' if p<0.01 else '**' if p<0.05 else '*' if p<0.1 else ''
        print(f'  {glabel:<12} β={b:>7.3f}{sig:<3} p={p:.4f}  R²={mod.rsquared:.4f}  N={int(mod.nobs)}')


Tenor: 1m
  Exporters    β=  0.250*** p=0.0052  R²=0.3173  N=4296
  Controls     β=  0.321*** p=0.0000  R²=0.4020  N=4296

Tenor: 3m
  Exporters    β=  0.070*** p=0.0001  R²=0.3172  N=4296
  Controls     β=  0.036**  p=0.0147  R²=0.4008  N=4296

Tenor: 6m
  Exporters    β=  0.031*** p=0.0006  R²=0.3170  N=4296
  Controls     β=  0.010    p=0.1389  R²=0.4007  N=4296

Tenor: 12m
  Exporters    β=  0.020*** p=0.0002  R²=0.3171  N=4296
  Controls     β=  0.008*** p=0.0067  R²=0.4007  N=4296

Tenor: 24m
  Exporters    β=  0.014*** p=0.0001  R²=0.3171  N=4296
  Controls     β=  0.007*** p=0.0000  R²=0.4008  N=4296


---

# 3. Jump Characterization

GARCH(1,1) vs. GARCH(1,1)-Jump likelihood ratio tests. The jump model nests the plain GARCH (set λ=0), so the LR statistic is χ²(3).

## 3.0 Estimation functions

In [12]:
@njit
def garch_loglik_numba(params, returns):
    """GARCH(1,1) log-likelihood."""
    mu, omega, alpha, beta = params
    T = len(returns)
    h = np.zeros(T)
    h[0] = np.var(returns)
    for t in range(1, T):
        h[t] = omega + alpha * (returns[t-1] - mu)**2 + beta * h[t-1]
    ll = 0.0
    for t in range(T):
        ll += -0.5 * (np.log(2 * np.pi * h[t]) + (returns[t] - mu)**2 / h[t])
    return -ll


def estimate_garch(returns):
    x0 = [np.mean(returns), 1e-5, 0.05, 0.90]
    bounds = [(None, None), (1e-8, None), (1e-6, 1.0), (1e-4, 0.999)]
    result = minimize(garch_loglik_numba, x0, args=(returns,), method='L-BFGS-B',
                      bounds=bounds, options={'maxiter': 2000})
    return {'params': result.x, 'loglik': -result.fun, 'converged': result.success}


@njit
def garch_jump_loglik_numba(params, returns, max_jumps=10):
    """GARCH(1,1)-Jump log-likelihood with Poisson jumps."""
    mu, omega, alpha, beta, lam, theta, delta = params
    T = len(returns)
    h = np.zeros(T)
    h[0] = np.var(returns)
    for t in range(1, T):
        h[t] = omega + alpha * (returns[t-1] - mu)**2 + beta * h[t-1]
    ll = 0.0
    for t in range(T):
        prob_t = 0.0
        log_poisson = -lam
        for k in range(max_jumps + 1):
            var_k = h[t] + k * delta**2
            mean_k = mu + k * theta
            pdf_k = np.exp(-0.5 * (returns[t] - mean_k)**2 / var_k) / np.sqrt(2 * np.pi * var_k)
            poisson_k = np.exp(log_poisson)
            prob_t += pdf_k * poisson_k
            log_poisson += np.log(lam) - np.log(k + 1)
        ll += np.log(prob_t + 1e-10)
    return -ll


def estimate_garch_jump(returns):
    x0 = [np.mean(returns), 1e-5, 0.05, 0.85, 0.03, -0.01, 0.03]
    bounds = [(None, None), (1e-8, None), (1e-6, 1.0), (1e-4, 0.999),
              (1e-4, 1.5), (-0.2, 0.2), (1e-4, 0.5)]
    result = minimize(garch_jump_loglik_numba, x0, args=(returns,), method='L-BFGS-B',
                      bounds=bounds, options={'maxiter': 2000})
    return {'params': result.x, 'loglik': -result.fun, 'converged': result.success}

## 3.1 Load daily data for jump estimation

In [14]:
# Daily data for jump tests (higher frequency = more power)
CDS_daily  = pd.read_csv('../data/processed/CDS/Daily_CDS.csv', parse_dates=['date'], index_col='date')
Oil_daily  = pd.read_csv('../data/processed/Oil/oil_prices_datastream.csv', parse_dates=['date'], index_col='date')
MSCI_daily = pd.read_csv('../data/processed/MSCI_indices/mscicountryindex.csv', parse_dates=['date'], index_col='date')

# Filter 2014+
CDS_daily  = CDS_daily[CDS_daily.index >= '2014-01-01']
Oil_daily  = Oil_daily[Oil_daily.index >= '2014-01-01']
MSCI_daily = MSCI_daily[MSCI_daily.index >= '2014-01-01']

# Returns
oil_returns  = np.log(Oil_daily / Oil_daily.shift(1)).dropna()
cds_returns  = np.log(CDS_daily / CDS_daily.shift(1)).dropna()
msci_returns = np.log(MSCI_daily / MSCI_daily.shift(1)).dropna()

print(f'Oil: {oil_returns.shape}, CDS: {cds_returns.shape}, MSCI: {msci_returns.shape}')

Oil: (2868, 4), CDS: (1970, 72), MSCI: (2869, 84)


In [15]:
# Country lists for jump tests
jump_countries = [
    'Brazil', 'Chile', 'China', 'Colombia', 'Egypt',
    'Indonesia', 'South Korea', 'Malaysia', 'Mexico',
    'Philippines', 'Qatar', 'Saudi Arabia', 'South Africa',
    'Thailand', 'Turkey', 'Abu Dhabi', 'Dubai'
]

jump_oil_exporters = [
    'Saudi Arabia', 'Qatar', 'Abu Dhabi', 'Dubai',
    'Colombia', 'Mexico', 'Brazil', 'Malaysia', 'Egypt'
]

## 3.2 Oil benchmarks

In [16]:
for benchmark in ['Brent', 'WTI', 'OPEC_basket', 'Dubai_Crude']:
    if benchmark not in Oil_daily.columns:
        continue
    oil_ret = Oil_daily[benchmark].pct_change().dropna().values

    garch     = estimate_garch(oil_ret)
    garch_jmp = estimate_garch_jump(oil_ret)

    T = len(oil_ret)
    LR = 2 * (garch_jmp['loglik'] - garch['loglik'])
    p  = 1 - chi2.cdf(LR, df=3)

    aic_g = -2*garch['loglik'] + 2*4
    aic_j = -2*garch_jmp['loglik'] + 2*7
    bic_g = -2*garch['loglik'] + 4*np.log(T)
    bic_j = -2*garch_jmp['loglik'] + 7*np.log(T)

    print(f'{benchmark}: LR={LR:.1f} p={p:.6f}  '
          f'AIC: {aic_g:.0f}→{aic_j:.0f}  BIC: {bic_g:.0f}→{bic_j:.0f}  '
          f'λ={garch_jmp["params"][4]:.3f} ({garch_jmp["params"][4]*252:.0f}/yr)  '
          f'θ={garch_jmp["params"][5]:.4f}  δ={garch_jmp["params"][6]:.4f}')

Brent: LR=289.5 p=0.000000  AIC: -14304→-14588  BIC: -14280→-14546  λ=0.193 (49/yr)  θ=-0.0066  δ=0.0253
WTI: LR=1061.5 p=0.000000  AIC: -12664→-13720  BIC: -12640→-13678  λ=0.065 (16/yr)  θ=-0.0160  δ=0.0379
OPEC_basket: LR=146.7 p=0.000000  AIC: -15158→-15298  BIC: -15134→-15257  λ=0.027 (7/yr)  θ=-0.0063  δ=0.0454
Dubai_Crude: LR=204.9 p=0.000000  AIC: -13997→-14196  BIC: -13973→-14154  λ=0.037 (9/yr)  θ=-0.0141  δ=0.0475


## 3.3 CDS series

In [17]:
cds_jump_results = []

for country in jump_countries:
    if country not in cds_returns.columns:
        print(f'{country}: NOT FOUND')
        continue
    ret = cds_returns[country].dropna().values
    if len(ret) < 200:
        continue
    if (ret == 0).sum() / len(ret) > 0.3:
        continue
    if np.var(ret) < 1e-10:
        continue

    garch     = estimate_garch(ret)
    garch_jmp = estimate_garch_jump(ret)
    T = len(ret)
    LR = 2 * (garch_jmp['loglik'] - garch['loglik'])
    pval = 1 - chi2.cdf(LR, df=3)

    cds_jump_results.append({
        'Country': country, 'T': T,
        'LR': LR, 'p_value': pval,
        'AIC_GARCH': -2*garch['loglik'] + 2*4,
        'AIC_Jump':  -2*garch_jmp['loglik'] + 2*7,
        'BIC_GARCH': -2*garch['loglik'] + 4*np.log(T),
        'BIC_Jump':  -2*garch_jmp['loglik'] + 7*np.log(T),
        'alpha_GARCH': garch['params'][2],
        'alpha_Jump':  garch_jmp['params'][2],
        'alpha_drop':  garch['params'][2] - garch_jmp['params'][2],
        'lambda': garch_jmp['params'][4],
        'theta':  garch_jmp['params'][5],
        'delta':  garch_jmp['params'][6],
        'Oil_Exporter': country in jump_oil_exporters,
    })
    sig = '***' if pval<0.01 else '**' if pval<0.05 else '*' if pval<0.1 else ''
    print(f'{country}: LR={LR:.1f} p={pval:.4f}{sig} λ={garch_jmp["params"][4]:.3f}')

cds_jump_df = pd.DataFrame(cds_jump_results)
print('\n', cds_jump_df.to_string())

Brazil: LR=208.0 p=0.0000*** λ=0.039
Chile: LR=338.5 p=0.0000*** λ=0.087
China: LR=475.6 p=0.0000*** λ=0.085
Colombia: LR=316.3 p=0.0000*** λ=0.081
Egypt: LR=2123.1 p=0.0000*** λ=0.030
Indonesia: LR=535.0 p=0.0000*** λ=0.342
South Korea: LR=557.5 p=0.0000*** λ=0.076
Malaysia: LR=1203.9 p=0.0000*** λ=1.500
Mexico: LR=271.5 p=0.0000*** λ=0.215
Philippines: LR=1276.9 p=0.0000*** λ=1.438
Qatar: LR=1191.8 p=0.0000*** λ=0.030
Saudi Arabia: LR=1160.8 p=0.0000*** λ=0.053
South Africa: LR=331.2 p=0.0000*** λ=1.103
Thailand: LR=2194.2 p=0.0000*** λ=0.908
Turkey: LR=437.0 p=0.0000*** λ=0.071
Abu Dhabi: LR=1698.8 p=0.0000*** λ=0.040
Dubai: LR=3513.1 p=0.0000*** λ=0.048

          Country     T           LR  p_value     AIC_GARCH      AIC_Jump     BIC_GARCH      BIC_Jump  alpha_GARCH  alpha_Jump  alpha_drop    lambda     theta     delta  Oil_Exporter
0         Brazil  1970   208.020024      0.0  -8910.414136  -9112.434160  -8888.070980  -9073.333638     0.066651    0.071851   -0.005200  0.038568  0

## 3.4 MSCI equity index series

In [18]:
# For MSCI, use original names (UAE instead of Abu Dhabi/Dubai)
msci_country_list = [
    'Brazil', 'Chile', 'China', 'Colombia', 'Egypt',
    'Indonesia', 'South Korea', 'Malaysia', 'Mexico',
    'Philippines', 'Qatar', 'Saudi Arabia', 'South Africa',
    'Thailand', 'Turkey', 'United Arab Emirates'
]

msci_jump_results = []

for country in msci_country_list:
    if country not in msci_returns.columns:
        print(f'{country}: NOT FOUND')
        continue
    ret = msci_returns[country].dropna().values
    if len(ret) < 200:
        continue
    if (ret == 0).sum() / len(ret) > 0.3:
        continue

    garch     = estimate_garch(ret)
    garch_jmp = estimate_garch_jump(ret)
    T = len(ret)
    LR = 2 * (garch_jmp['loglik'] - garch['loglik'])
    pval = 1 - chi2.cdf(LR, df=3)

    is_oil = country in jump_oil_exporters or country == 'United Arab Emirates'

    msci_jump_results.append({
        'Country': country, 'T': T,
        'LR': LR, 'p_value': pval,
        'AIC_GARCH': -2*garch['loglik'] + 2*4,
        'AIC_Jump':  -2*garch_jmp['loglik'] + 2*7,
        'lambda': garch_jmp['params'][4],
        'theta':  garch_jmp['params'][5],
        'delta':  garch_jmp['params'][6],
        'alpha_drop': garch['params'][2] - garch_jmp['params'][2],
        'Oil_Exporter': is_oil,
    })
    sig = '***' if pval<0.01 else '**' if pval<0.05 else '*' if pval<0.1 else ''
    print(f'{country}: LR={LR:.1f} p={pval:.4f}{sig} λ={garch_jmp["params"][4]:.3f}')

msci_jump_df = pd.DataFrame(msci_jump_results)
print('\n', msci_jump_df.to_string())

Brazil: LR=132.8 p=0.0000*** λ=0.037
Chile: LR=110.4 p=0.0000*** λ=0.037
China: LR=102.7 p=0.0000*** λ=0.039
Colombia: LR=153.9 p=0.0000*** λ=0.045
Egypt: LR=1940.2 p=0.0000*** λ=0.045
Indonesia: LR=153.4 p=0.0000*** λ=0.030
South Korea: LR=65.8 p=0.0000*** λ=0.030
Malaysia: LR=151.1 p=0.0000*** λ=0.030
Mexico: LR=114.2 p=0.0000*** λ=0.031
Philippines: LR=112.2 p=0.0000*** λ=0.030
Qatar: LR=578.6 p=0.0000*** λ=0.030
Saudi Arabia: LR=769.8 p=0.0000*** λ=0.030
South Africa: LR=86.3 p=0.0000*** λ=0.036
Thailand: LR=158.8 p=0.0000*** λ=0.030
Turkey: LR=353.1 p=0.0000*** λ=0.038
United Arab Emirates: LR=480.8 p=0.0000*** λ=0.030

                  Country     T           LR       p_value     AIC_GARCH      AIC_Jump    lambda     theta     delta    alpha_drop  Oil_Exporter
0                 Brazil  2869   132.791416  0.000000e+00 -14734.019749 -14860.811165  0.037313 -0.012789  0.048495  2.055188e-03          True
1                  Chile  2869   110.439556  0.000000e+00 -16520.833234 -16625

## 3.5 Group comparisons

In [19]:
for label, df in [('CDS', cds_jump_df), ('MSCI', msci_jump_df)]:
    print(f'\n{"="*60}')
    print(f'{label} — Mean jump parameters by group:')
    print(df.groupby('Oil_Exporter')[['LR', 'lambda', 'theta', 'delta', 'alpha_drop']].mean())

    exp  = df[df['Oil_Exporter']]
    ctrl = df[~df['Oil_Exporter']]

    for metric in ['LR', 'lambda', 'alpha_drop']:
        t, p = ttest_ind(exp[metric].dropna(), ctrl[metric].dropna())
        u, p_mw = mannwhitneyu(exp[metric].dropna(), ctrl[metric].dropna(), alternative='greater')
        print(f'  {metric}: Exp={exp[metric].mean():.4f} Ctrl={ctrl[metric].mean():.4f} '
              f't={t:.2f} p={p:.4f}  MW-U={u:.0f} p={p_mw:.4f}')


CDS — Mean jump parameters by group:
                       LR    lambda     theta     delta  alpha_drop
Oil_Exporter                                                       
False          768.239481  0.513893  0.014193  0.041943    0.014677
True          1298.594919  0.226192  0.001725  0.054778    0.021066
  LR: Exp=1298.5949 Ctrl=768.2395 t=1.22 p=0.2403  MW-U=40 p=0.3715
  lambda: Exp=0.2262 Ctrl=0.5139 t=-1.15 p=0.2691  MW-U=14 p=0.9863
  alpha_drop: Exp=0.0211 Ctrl=0.0147 t=0.21 p=0.8400  MW-U=33 p=0.6285

MSCI — Mean jump parameters by group:
                      LR    lambda     theta    delta  alpha_drop
Oil_Exporter                                                     
False         142.843726  0.033896 -0.009990  0.03727   -0.002865
True          540.165880  0.034795 -0.011701  0.03464    0.002011
  LR: Exp=540.1659 Ctrl=142.8437 t=1.80 p=0.0931  MW-U=53 p=0.0141
  lambda: Exp=0.0348 Ctrl=0.0339 t=0.32 p=0.7533  MW-U=32 p=0.5204
  alpha_drop: Exp=0.0020 Ctrl=-0.0029 t=0.52 p

## 3.6 Co-jump analysis (Oil × CDS)

In [20]:
@njit
def get_garch_variance(params, returns):
    mu, omega, alpha, beta = params
    T = len(returns)
    h = np.zeros(T)
    h[0] = np.var(returns)
    for t in range(1, T):
        h[t] = omega + alpha * (returns[t-1] - mu)**2 + beta * h[t-1]
    return h


def identify_jumps(returns, threshold=2.5):
    ret = returns.dropna().values
    garch = estimate_garch(ret)
    h = get_garch_variance(garch['params'], ret)
    mu = garch['params'][0]
    z = (ret - mu) / np.sqrt(h)
    return pd.Series(np.abs(z) > threshold, index=returns.dropna().index)


def cojump_test(oil_jumps, cds_jumps, country):
    common_idx = oil_jumps.index.intersection(cds_jumps.index)
    oil_j = oil_jumps.loc[common_idx].values
    cds_j = cds_jumps.loc[common_idx].values
    n = len(common_idx)
    a = ((oil_j) & (cds_j)).sum()
    b = ((~oil_j) & (cds_j)).sum()
    c = ((oil_j) & (~cds_j)).sum()
    d = ((~oil_j) & (~cds_j)).sum()
    n_oil = oil_j.sum()
    odds_ratio, fisher_p = fisher_exact([[a, b], [c, d]])
    cojump_rate = a / n_oil if n_oil > 0 else np.nan
    return {'Country': country, 'N': n, 'N_oil_jumps': int(n_oil),
            'N_cds_jumps': int(cds_j.sum()), 'N_cojumps': int(a),
            'Cojump_rate': cojump_rate, 'Odds_ratio': odds_ratio, 'Fisher_p': fisher_p}

In [21]:
brent_returns = oil_returns['Brent']
oil_jumps = identify_jumps(brent_returns, threshold=2.5)
print(f'Oil (Brent) jumps: {oil_jumps.sum()} / {len(oil_jumps)} ({oil_jumps.mean()*100:.1f}%)')

cojump_results = []
for country in jump_countries:
    if country not in cds_returns.columns:
        continue
    ret = cds_returns[country].dropna()
    if len(ret) < 200 or (ret == 0).sum()/len(ret) > 0.3:
        continue
    cds_j = identify_jumps(ret, threshold=2.5)
    cojump_results.append(cojump_test(oil_jumps, cds_j, country))

cojump_df = pd.DataFrame(cojump_results)
cojump_df['Oil_Exporter'] = cojump_df['Country'].isin(jump_oil_exporters)
print('\n', cojump_df.to_string(index=False))

# Group comparison
print('\nMean co-jump rate by group:')
print(cojump_df.groupby('Oil_Exporter')[['Cojump_rate', 'Odds_ratio']].mean())

exp_cj  = cojump_df[cojump_df['Oil_Exporter']]['Cojump_rate'].dropna()
ctrl_cj = cojump_df[~cojump_df['Oil_Exporter']]['Cojump_rate'].dropna()
t, p = ttest_ind(exp_cj, ctrl_cj)
u, p_mw = mannwhitneyu(exp_cj, ctrl_cj, alternative='greater')
print(f'\nT-test: t={t:.3f} p={p:.4f}')
print(f'MW-U:   U={u:.0f} p={p_mw:.4f}')

Oil (Brent) jumps: 77 / 2868 (2.7%)

      Country    N  N_oil_jumps  N_cds_jumps  N_cojumps  Cojump_rate  Odds_ratio  Fisher_p  Oil_Exporter
      Brazil 1968           48           50          4     0.083333    3.703557  0.031539          True
       Chile 1968           48           52          4     0.083333    3.545455  0.035784         False
       China 1968           48           51          2     0.041667    1.660160  0.355288         False
    Colombia 1968           48           55          4     0.083333    3.331551  0.042751          True
       Egypt 1968           48           57          4     0.083333    3.202401  0.047796          True
   Indonesia 1968           48           58          4     0.083333    3.141414  0.050439         False
 South Korea 1968           48           66          3     0.062500    1.965079  0.216002         False
    Malaysia 1968           48           57          4     0.083333    3.202401  0.047796          True
      Mexico 1968         